# 🧪 XNLI (Spanish) — NLI with BETO — **STUDENT VERSION (Guided)**
You will build a **Natural Language Inference (NLI)** model in Spanish on **XNLI (es)** using **BETO** (`dccuchile/bert-base-spanish-wwm-cased`).
**Labels:** `entailment (0)`, `neutral (1)`, `contradiction (2)`.

This version removes more code and replaces it with **detailed TODOs** that include **hints** and **mini-checklists**.
Follow the TODOs carefully. Each one tells you **what** to do and gives you **clues** about the missing code.


## ✅ Learning Goals
- Load & explore **XNLI (es)**.
- Tokenize **(premise, hypothesis)** pairs for BERT-style models.
- Fine-tune **BETO** for 3-way NLI with the Hugging Face **Trainer**.
- Evaluate with **accuracy/precision/recall/F1-macro**; draw a **confusion matrix**.
- Do **error analysis** (inspect a few mistakes).
- Save & reload the best model; write a tiny **inference** helper.


## 🧰 Quick Start (Colab)
1. Run **Cell 0** below (Reset & Install).
2. **Runtime → Restart**.
3. Continue from **Section 1** (skip Cell 0 after restart).


## 🔁 Cell 0 — Colab Reset & Install (run once, then restart)
Pins a stable stack and avoids NumPy ABI conflicts.


In [ ]:
# ✅ Run this cell ONCE, then restart the runtime.
import sys
pip = f"{sys.executable} -m pip"

!{pip} uninstall -y transformers tokenizers datasets accelerate evaluate simpletransformers numpy
!{pip} install -U "numpy==2.0.2"
!{pip} install -U "pyarrow-hotfix" "fsspec==2024.3.1"
!{pip} install -U "transformers==4.56.2" "datasets==2.19.1" "accelerate==1.10.1" "evaluate==0.4.1" "scikit-learn"

print("✅ Done. Now: Runtime → Restart. Then continue from Section 1.")


## 1) Setup & Config
**What you will do:**
- Import only the required libraries.
- Set device, random seed, model name, run name, and hyperparameters.
- Create output folders.

**Hints & checklist:**
- Use `torch.cuda.is_available()` to choose `"cuda"` or `"cpu"`.
- Fix a `SEED` (e.g., 42) and seed Python `random`, `numpy`, and `torch` (+ `cuda` if available).
- Suggested defaults: `MODEL_NAME="dccuchile/bert-base-spanish-wwm-cased"`, `LR=2e-5`, `EPOCHS=3`, `BATCH_TRAIN=16`, `BATCH_EVAL=32`, `WEIGHT_DECAY=0.01`.


In [ ]:
# TODO 1: Imports & basic config
# (a) Import: os, random, json, time (optional), numpy as np, torch
# (b) Import HF: load_dataset, AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
# (c) Import metrics/plotting: accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, matplotlib.pyplot as plt

# --- Your code below ---


# TODO 1.1: Set DEVICE, print it


# TODO 1.2: Seed Python, numpy, torch (and torch.cuda if available)


# TODO 1.3: Set MODEL_NAME, RUN_NAME, LR, EPOCHS, BATCH_TRAIN, BATCH_EVAL, WEIGHT_DECAY


# TODO 1.4: Create OUTPUT_DIR and BEST_DIR; os.makedirs(..., exist_ok=True)


## 2) Data — Load XNLI (Spanish subset)
**What you will do:**
- Load the dataset and print basic info (splits/sizes, one example).

**Hints & checklist:**
- `raw = load_dataset("xnli", "es")`
- Keys: `"train"`, `"validation"`, `"test"`
- Fields per item: `premise`, `hypothesis`, `label`.


In [ ]:
# TODO 2: Load XNLI (es) and inspect
# --- Your code below ---


## 3) Labels & Tokenization
**What you will do:**
- Ensure `labels` exists (Trainer expects this name).
- Load tokenizer and tokenize `(premise, hypothesis)` pairs with truncation.
- Keep text columns for error analysis.
- Use dynamic padding.

**Hints & checklist:**
- `raw = raw.map(lambda b: {"labels": b["label"]})`
- `tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)`
- In `tokenize_fn`, call tokenizer on `(premise, hypothesis)` with `truncation=True` and then attach labels.
- Map over each split to produce `train_ds`, `val_ds`, `test_ds`.
- `data_collator = DataCollatorWithPadding(tokenizer=tokenizer)`


In [ ]:
# TODO 3: Prepare labels + tokenize + collator
# --- Your code below ---


## 4) Model — BETO for 3-way classification
**What you will do:**
- Load BETO as a classifier with `num_labels=3` and move it to DEVICE.


In [ ]:
# TODO 4: Model
# NUM_LABELS = 3
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
# model.to(DEVICE)

# --- Your code below ---


## 5) Metrics
**What you will do:**
- Implement `compute_metrics` returning accuracy, precision, recall, f1 (macro).

**Hints:**
- Handle both `(logits, labels)` tuple and `.predictions/.label_ids` object.
- `preds = argmax(logits, axis=-1)`.
- Use `precision_recall_fscore_support(..., average="macro", zero_division=0)`.


In [ ]:
# TODO 5: compute_metrics
# --- Your code below ---


## 6) Training (`TrainingArguments` + `Trainer`)
**What you will do:**
- Fill `TrainingArguments` (see typical args below).
- Build `Trainer` and train.
- Print best checkpoint; evaluate on validation.

**Typical args to include:**
- `output_dir=OUTPUT_DIR`
- `per_device_train_batch_size=BATCH_TRAIN`, `per_device_eval_batch_size=BATCH_EVAL`
- `learning_rate=LR`, `num_train_epochs=EPOCHS`, `weight_decay=WEIGHT_DECAY`
- `evaluation_strategy="epoch"`, `save_strategy="epoch"`
- `load_best_model_at_end=True`, `metric_for_best_model="f1"`, `greater_is_better=True`
- Plus QoL: `save_total_limit=2`, `logging_steps=50`, `report_to="none"`, `fp16=torch.cuda.is_available()`


In [ ]:
# TODO 6: Train and validate
# (a) args = TrainingArguments(...)
# (b) trainer = Trainer(...)
# (c) train_result = trainer.train() ; print(train_result)
# (d) print best checkpoint
# (e) val_metrics = trainer.evaluate(eval_dataset=val_ds)

# --- Your code below ---


## 7) Test Evaluation
**What you will do:**
- Predict on test, compute metrics, and draw a confusion matrix (single Matplotlib plot).

**Hints:**
- `pred = trainer.predict(test_ds)`; use `pred.predictions` and `pred.label_ids`.
- `y_pred = argmax(logits, axis=-1)`.


In [ ]:
# TODO 7: Test metrics + confusion matrix
# --- Your code below ---


## 8) Error Analysis
**What you will do:**
- Show up to 10 misclassified examples (gold vs pred, premise & hypothesis).


In [ ]:
# TODO 8: Print up to 10 mistakes
# --- Your code below ---


## 9) Save, Reload, and Inference
**What you will do:**
- Save the best model + tokenizer to `BEST_DIR`, reload them, and implement `infer_nli`.

**Hints:**
- Reload with `AutoTokenizer.from_pretrained(BEST_DIR)` and `AutoModelForSequenceClassification.from_pretrained(BEST_DIR)`.
- Use a numerically stable softmax for probabilities.


In [ ]:
# TODO 9: Save, reload, and write an inference helper
# --- Your code below ---


## 10) ✅ What to Submit
- Your **completed notebook** with all cells executed (outputs visible).
- Brief answers:
  1) Which class is hardest, and why?
  2) Two common error patterns.
  3) One improvement idea (e.g., hyperparams, early stopping, class weights, data cleaning).


## 11) Troubleshooting
- NumPy ABI / `_center` import errors → re-run **Cell 0**, then **Restart**.
- `evaluation_strategy` not recognized → use **Cell 0** (installs Transformers 4.56.2).
- Model returned only logits (no loss) → ensure your tokenization adds a `labels` field.
- CUDA OOM → lower batch sizes or use gradient accumulation.
- Keep your `SEED` fixed for reproducibility.
